# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Lourdes Tolotto**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [3]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

## **Búsqueda A\* usando heurística**
Definicion de la funcion



In [ ]:
def heuristic(state):
    goal_rod = state.rods[2]                            # La tercera torre es el objetivo
    return state.number_of_disks - len(goal_rod)        # Cuántos discos faltan para completar el objetivo


<h2 style="color:#27ae60;">Algoritmo de Búsqueda A* </h2>

Este código implementa un **algoritmo de búsqueda heurística** para resolver el problema de las Torres de Hanoi con un número dado de discos.

## Flujo general

1. **Definir el problema**
   - Se crea la lista de discos iniciales `[n, n-1, ..., 1]`.
   - Se establece el **estado inicial**: todos los discos en la primera torre.
   - Se define el **estado objetivo**: todos los discos en la tercera torre.
   - Se instancia un objeto `ProblemHanoi` que contiene el estado inicial y el objetivo.

2. **Nodo raíz y frontera**
   - Se crea un nodo raíz (`NodeHanoi`) con el estado inicial.
   - La **frontera** es una lista que funciona como cola de prioridad según el costo total `f = g + h`.
   - El nodo raíz se agrega a la frontera con su valor heurístico.

3. **Exploración de nodos**
   - Mientras haya nodos en la frontera:
     - Se extrae el nodo con menor costo total.
     - Si el estado ya fue visitado, se ignora.
     - Se marca el estado como visitado.
     - Se actualiza la profundidad máxima alcanzada.
     - Se verifica si el nodo actual cumple el **estado objetivo**:
       - Si ocurre, se guarda como solución y se detiene la búsqueda.
     - Si no, se generan los **nodos hijos** aplicando todas las acciones posibles:
       - Para cada hijo se calcula `g` (costo acumulado), `h` (heurística) y `f = g + h`.
       - Se agregan los hijos a la frontera.

4. **Resultados**
   - Se determina si se encontró solución.
   - Se calculan métricas de desempeño:
     - Número de nodos explorados
     - Número de estados visitados
     - Profundidad máxima
     - Costo total de la solución
     - Tamaño actual de la frontera


In [19]:
import heapq

def search_algorithm(number_disks=5) -> (NodeHanoi, dict):

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)                 # Crear la lista de discos [5,4,3,2,1] 
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)                    # El objetivo es tener todos los discos en la tercera torre
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)                          # Crear el problema de Hanoi con el estado inicial y el estado objetivo

    root = NodeHanoi(state=initial_state)                                                   # Nodo raíz del árbol de búsqueda con el estado inicial (sin padre ni acción previa)

    frontier = []                                                                           # Frontera vacia, se usará una cola de prioridad basada en el costo total (g + h)
    heapq.heappush(frontier, (heuristic(initial_state), root))                              # Agregar el nodo raíz a la frontera con su valor heurístico como prioridad

    visited_states = set()                                                                  # Conjunto para rastrear los estados visitados y evitar ciclos 

    nodes_explored = 0
    max_depth = 0
    solution = None

    while frontier:                                                                         # Mientras haya nodos en la frontera el algoritmo continúa explorando

        _, node = heapq.heappop(frontier)                                                   # Extraer el nodo con el menor costo total (g + h) de la frontera
        nodes_explored += 1

        state = node.state

        if state in visited_states:                                                         # Si el estado ya ha sido visitado, se omite para evitar ciclos
            continue                                                                        # Se vuelve al inicio del bucle para procesar el siguiente nodo en la frontera

        visited_states.add(state)                                                           # Marcar el estado actual como visitado si no se ha visitado antes

        max_depth = max(max_depth, node.depth)                                              # Actualizar la profundidad máxima alcanzada en la búsqueda                    

        if problem.goal_test(state):                                                        # Verificar si el estado actual es el estado objetivo
            solution = node                                                                 # Si ocurre, se guarda el nodo solución y se sale del bucle de búsqueda
            break

        for action in problem.actions(state):                                               # Obtener las acciones posibles desde el estado actual

            child = node.child_node(problem, action)                                        # Generar el nodo hijo resultante de aplicar la acción al estado actual

            g = child.path_cost                                                             # Costo acumulado para llegar al nodo hijo desde el nodo raíz
            h = heuristic(child.state)                                                      # Costo estimado para llegar al objetivo desde el nodo hijo usando la función heurística             
            f = g + h                                                                       # Calcular el costo total (g + h) para el nodo hijo

            heapq.heappush(frontier, (f, child))                                            # Agregar el nodo hijo a la frontera ordenado por prioridad

    # Finaliza la búsqueda, se ha encontrado una solución o se han agotado los nodos en la frontera
    
    solution_found = solution is not None

    if solution_found:                                                                      # Si hay una solución, se calcula el costo total desde el nodo raíz hasta el nodo solución
        cost_total = solution.path_cost
    else:
        cost_total = None

    metrics = {
        "solution_found": solution_found,
        "nodes_explored": nodes_explored,
        "states_visited": len(visited_states),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": cost_total,
    }

    return solution, metrics


Se ejecuta la función y se muestra el resultado de la solución:

In [20]:
solution, metrics = search_algorithm(number_disks=5)
print("Solución encontrada:", solution is not None)

Solución encontrada: True


Se muestran las métricas:

In [18]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 496
states_visited: 178
nodes_in_frontier: 35
max_depth: 31
cost_total: 31.0


Camino de estados desde el principio a la solución:

In [13]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Acciones aplicadas para llegar al objetivo:

In [14]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3


<h2 style="color:#27ae60;">Conclusiones (A* con 5 discos)</h2>

Se resolvió el problema de **Torres de Hanoi** utilizando el algoritmo **A*** con un total de **5 discos**.

---

### Costo de la solución

Para el problema de Torres de Hanoi, la cantidad mínima de movimientos necesaria para resolverlo está dada por:
[
2^n - 1
]

donde **n** es el número de discos. Para **5 discos**:
[
2^5 - 1 = 31
]

El algoritmo encontró una solución con:

```
cost_total: 31
```

Esto indica que **A*** logró encontrar la **solución óptima**, es decir, la secuencia mínima de movimientos necesaria para trasladar todos los discos desde la primera torre hasta la tercera.
A diferencia de algoritmos como **DFS**, el algoritmo **A*** utiliza una función heurística:
[
f(n) = g(n) + h(n)
]

donde:

* **g(n)** representa el costo acumulado desde el estado inicial hasta el nodo actual.
* **h(n)** es una estimación del costo restante hasta el estado objetivo.

Esta combinación permite que el algoritmo explore primero los caminos más prometedores y garantiza encontrar la solución óptima cuando la heurística utilizada es admisible.

---

### Profundidad alcanzada

La métrica obtenida fue:

```
max_depth: 31
```

Esto indica que el nodo objetivo se encontró a una profundidad de **31 niveles** en el árbol de búsqueda.
Como cada movimiento tiene un costo de **1**, la profundidad coincide exactamente con el costo total de la solución óptima.

---

### Exploración del espacio de estados

Las métricas obtenidas fueron:

```
nodes_explored: 496
states_visited: 178
nodes_in_frontier: 35
```

**Interpretación:**

* **nodes_explored:** cantidad total de nodos que el algoritmo extrajo de la frontera para expandir.
* **states_visited:** número de estados únicos visitados durante la búsqueda.
* **nodes_in_frontier:** nodos que permanecían en la frontera cuando se encontró la solución.

Para el caso de **5 discos**, el espacio total de estados posibles es:
[
3^5 = 243
]

ya que cada disco puede encontrarse en cualquiera de las tres torres.

El algoritmo visitó **178 estados únicos**, lo que indica que exploró una porción significativa del espacio de estados, aunque no fue necesario recorrerlo completamente gracias al uso de la heurística.

La diferencia entre **nodes_explored** y **states_visited** se debe a que múltiples caminos pueden generar el mismo estado. Estos casos se controlan mediante una estructura `set` que evita volver a explorar configuraciones ya visitadas.

---

### Conclusión general

El algoritmo **A*** logró resolver correctamente el problema encontrando la **solución óptima de 31 movimientos**.

Aunque se exploraron varios cientos de nodos durante la búsqueda, el uso de una heurística permitió guiar la exploración del árbol de estados, reduciendo la cantidad de configuraciones que debieron analizarse para alcanzar la solución.



<h2 style="color:#27ae60;">Estructura del problema: Clases principales</h2>

La implementación del problema de **:contentReference[oaicite:0]{index=0}** se basa en tres clases principales:

- `StatesHanoi`
- `ActionHanoi`
- `ProblemHanoi`

Cada una representa un componente fundamental del modelo de búsqueda utilizado por el algoritmo.

---

# Clase `StatesHanoi`

Representa **un estado del problema**, es decir, la configuración actual de los discos en las tres varillas.

## Atributos principales

- **rods**  
  Lista con las tres varillas del problema.  
  Cada varilla es una lista que contiene los discos en orden.

- **number_of_disks**  
  Cantidad total de discos presentes en el estado.

- **number_of_pegs**  
  Cantidad de varillas del problema (en este caso siempre 3).

- **accumulated_cost**  
  Costo acumulado desde el estado inicial hasta el estado actual.

- **__string_representation__**  
  Representación en forma de string del estado, utilizada para mostrarlo y para generar el hash.

## Métodos principales

- **`__init__()`**  
  Inicializa el estado verificando que la configuración sea válida:
  - no haya discos repetidos en diferentes varillas
  - los discos tengan valores correctos
  - todos los discos estén presentes
  - cada varilla esté ordenada correctamente

- **`get_last_disk_rod()`**  
  Obtiene el último disco de una varilla específica.  
  Puede devolverlo o solo observarlo (`peek=True`).

- **`check_valid_disk_in_rod()`**  
  Verifica si un disco puede colocarse en una varilla respetando las reglas del problema.

- **`put_disk_in_rod()`**  
  Coloca un disco en una varilla si la acción es válida.

- **`accumulate_cost()`**  
  Suma un costo al costo acumulado del estado.

- **`get_accumulated_cost()`**  
  Devuelve el costo acumulado del estado.

- **`get_state()`**  
  Devuelve la representación del estado como lista de varillas.

- **`get_state_dict()`**  
  Devuelve el estado en formato diccionario.

## Métodos especiales

- `__eq__()` → compara dos estados  
- `__hash__()` → permite usar el estado en estructuras como `set`  
- `__str__()` y `__repr__()` → representación en texto del estado  

Estos métodos permiten que los estados puedan ser utilizados en algoritmos de búsqueda evitando repetir estados ya visitados.

---

# Clase `ActionHanoi`

Representa **una acción posible en el problema**, es decir, el movimiento de un disco entre varillas.

## Atributos principales

- **disk**  
  Número del disco que se mueve.

- **rod_input**  
  Varilla desde la que se mueve el disco.

- **rod_out**  
  Varilla hacia la que se mueve el disco.

- **action**  
  Descripción en texto del movimiento.

- **action_dict**  
  Representación de la acción en formato diccionario.

- **cost**  
  Costo asociado a la acción (en este caso, mover un disco cuesta 1).

## Métodos principales

- **`__init__()`**  
  Inicializa la acción y define su representación textual y su costo.

- **`execute()`**  
  Ejecuta la acción sobre un estado dado:
  1. copia el estado
  2. mueve el disco correspondiente
  3. actualiza el costo acumulado
  4. devuelve el nuevo estado

## Métodos especiales

- `__str__()`  
- `__repr__()`

Permiten mostrar fácilmente la acción realizada.

---

# Clase `ProblemHanoi`

Define formalmente el problema de búsqueda siguiendo la estructura de **:contentReference[oaicite:1]{index=1}**, extendiendo la clase `Problem`.

## Atributos principales

- **initial**  
  Estado inicial del problema.

- **goal**  
  Estado objetivo del problema.

## Métodos principales

- **`actions(state)`**  
  Devuelve todas las acciones válidas que pueden ejecutarse desde un estado dado.

- **`result(state, action)`**  
  Aplica una acción a un estado y devuelve el nuevo estado resultante.

- **`path_cost(c, state1, action, state2)`**  
  Calcula el costo total del camino sumando el costo acumulado del estado anterior y el costo de la acción ejecutada.

---

# Rol de las clases en el algoritmo de búsqueda

| Clase | Rol |
|------|------|
| `StatesHanoi` | Representa configuraciones del problema |
| `ActionHanoi` | Representa movimientos posibles |
| `ProblemHanoi` | Define reglas del problema para el algoritmo de búsqueda |

